In [4]:
# =========================================
# BOOK RECOMMENDATION SYSTEM 
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [5]:
# =========================
# 2. LOAD DATA
# =========================
# IMPORTANT:
# Your file has .xls name, but it is actually CSV text
df = pd.read_csv("books (2).xls")

print("Shape of dataset:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nFirst 5 rows:")
display(df.head())



Shape of dataset: (6810, 12)

Columns:
Index(['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories',
       'thumbnail', 'description', 'published_year', 'average_rating',
       'num_pages', 'ratings_count'],
      dtype='object')

First 5 rows:


,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [6]:
# =========================
# 3. BASIC CLEANING
# =========================
df.columns = df.columns.str.strip()

# Select useful columns
books = df[['title', 'authors', 'categories', 'description', 'thumbnail', 'average_rating', 'ratings_count']].copy()

# Fill missing values
books['title'] = books['title'].fillna('')
books['authors'] = books['authors'].fillna('')
books['categories'] = books['categories'].fillna('')
books['description'] = books['description'].fillna('')
books['thumbnail'] = books['thumbnail'].fillna('')
books['average_rating'] = books['average_rating'].fillna(0)
books['ratings_count'] = books['ratings_count'].fillna(0)

# Remove duplicate titles
books = books.drop_duplicates(subset='title').reset_index(drop=True)

print("Shape after cleaning:", books.shape)
display(books.head())



Shape after cleaning: (6398, 7)


,title,authors,categories,description,thumbnail,average_rating,ratings_count
0,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,361.0
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,5164.0
2,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,172.0
3,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,29532.0
4,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,33684.0


In [7]:
# =========================
# 4. TEXT CLEANING FUNCTION
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

books['title_clean'] = books['title'].apply(clean_text)
books['authors_clean'] = books['authors'].apply(clean_text)
books['categories_clean'] = books['categories'].apply(clean_text)
books['description_clean'] = books['description'].apply(clean_text)



In [8]:
# =========================
# 5. CREATE TAGS
# =========================
books['tags'] = (
    books['title_clean'] + ' ' +
    books['authors_clean'] + ' ' +
    books['categories_clean'] + ' ' +
    books['description_clean']
)

display(books[['title', 'tags']].head())



,title,tags
0,Gilead,gilead marilynne robinson fiction a novel that...
1,Spider's Web,spider s web charles osborne agatha christie d...
2,The One Tree,the one tree stephen r donaldson american fict...
3,Rage of angels,rage of angels sidney sheldon fiction a memora...
4,The Four Loves,the four loves clive staples lewis christian l...


In [9]:
# =========================
# 6. TF-IDF VECTORIZATION
# =========================
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(books['tags'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)



TF-IDF matrix shape: (6398, 5000)


In [10]:
# =========================
# 7. COSINE SIMILARITY
# =========================
similarity = cosine_similarity(tfidf_matrix)
print("Similarity matrix shape:", similarity.shape)



Similarity matrix shape: (6398, 6398)


In [11]:
# =========================
# 8. RECOMMENDATION FUNCTION
# =========================
def recommend(book_name, top_n=5):
    book_name = book_name.lower()
    
    matched_books = books[books['title'].str.lower() == book_name]
    
    if matched_books.empty:
        print("Book not found in dataset.")
        return None
    
    book_index = matched_books.index[0]
    distances = similarity[book_index]
    
    book_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:top_n+1]
    
    recommendations = []
    
    for i in book_list:
        idx = i[0]
        recommendations.append({
            'title': books.iloc[idx]['title'],
            'authors': books.iloc[idx]['authors'],
            'category': books.iloc[idx]['categories'],
            'rating': books.iloc[idx]['average_rating'],
            'ratings_count': books.iloc[idx]['ratings_count'],
            'thumbnail': books.iloc[idx]['thumbnail']
        })
    
    return pd.DataFrame(recommendations)



In [12]:
# =========================
# 9. TEST THE SYSTEM
# =========================
book_to_test = books['title'].iloc[0]
print("Testing recommendation for:", book_to_test)

result = recommend(book_to_test, top_n=5)
display(result)



Testing recommendation for: Gilead


,title,authors,category,rating,ratings_count,thumbnail
0,Go Tell it on the Mountain,James Baldwin,Fiction,4.01,33558.0,http://books.google.com/books/content?id=eyg78...
1,Four Baboons Adoring the Sun,John Guare,Drama,4.00,234.0,http://books.google.com/books/content?id=FXvPP...
2,The Deep End of the Ocean,Jacquelyn Mitchard,Fiction,3.86,110552.0,http://books.google.com/books/content?id=247V7...
3,Children of the Alley,Najīb Maḥfūẓ,Fiction,4.10,1124.0,http://books.google.com/books/content?id=1XmQt...
4,The Languages of Pao,Jack Vance,Fiction,3.80,862.0,http://books.google.com/books/content?id=bQkHA...


In [13]:
# =========================
# 10. SEARCH HELPER FUNCTION
# =========================
def search_books(keyword, n=10):
    keyword = keyword.lower()
    result = books[books['title'].str.lower().str.contains(keyword, na=False)]
    return result[['title', 'authors', 'categories', 'average_rating']].head(n)

# Example search
print("Search results:")
display(search_books("harry"))



Search results:


,title,authors,categories,average_rating
1596,The Harry Bosch Novels,Michael Connelly,Fiction,4.34
1611,"Harry Bosch Novels, The: Volume 2",Michael Connelly,Fiction,4.48
2558,Harry Potter and the Chamber of Secrets (Book 2),"Rowling, J.K.",Juvenile Fiction,4.41
2573,Harry Potter and the Order of the Phoenix (Boo...,"Rowling, J.K.",Juvenile Fiction,4.49
2593,Harry Potter and the Chamber of Secrets,J. K. Rowling;Mary GrandPre,Juvenile Fiction,4.41
2594,Harry Potter and the Sorcerer's Stone (Book 1),"Rowling, J.K.",Juvenile Fiction,4.47
2606,Harry Potter and the Prisoner of Azkaban (Book 3),"Rowling, J.K.",Juvenile Fiction,4.55
2608,Harry Potter,J. K. Rowling,Juvenile Fiction,4.78
2619,Harry Potter and the Half-Blood Prince (Book 6),"Rowling, J.K.",Juvenile Fiction,4.56
2626,The Harry Potter Collection,J. K. Rowling,Juvenile Fiction,4.73


In [14]:
# =========================
# 11. SAVE FILES FOR STREAMLIT
# =========================
book_dict = books[['title', 'authors', 'categories', 'description', 'thumbnail', 'average_rating', 'ratings_count']].to_dict()

with open("books_dict.pkl", "wb") as f:
    pickle.dump(book_dict, f)

with open("book_similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

print("Files saved successfully:")
print("1. books_dict.pkl")
print("2. book_similarity.pkl")

Files saved successfully:
1. books_dict.pkl
2. book_similarity.pkl
